# FT-Transformer: tutorial e otimização de hiperparâmetros

**FT-Transformer** (Gorishniy et al., 2021, *Revisiting Deep Learning Models for Tabular Data*): cada feature (numérica ou categórica) vira um *token* via embedding; um token `[CLS]` é adicionado, passa por blocos Transformer, e sua saída final alimenta a camada de predição.

**Bibliotecas existentes**
| Biblioteca | Observação |
|---|---|
| [`rtdl_revisiting_models`](https://github.com/yandex-research/rtdl-revisiting-models) | Implementação oficial (Yandex), PyTorch puro, mínima. (Estamos usando aqui) |
| [`pytorch_tabular`](https://github.com/manujosephv/pytorch_tabular) | Framework completo (Lightning) com `FTTransformerConfig`. |
| [`tab-transformer-pytorch`](https://github.com/lucidrains/tab-transformer-pytorch) | Implementação compacta (lucidrains). |

Dataset sintético apenas para demonstração; o foco são os hiperparâmetros (o que podemos otimizar).

In [1]:
# !pip install rtdl_revisiting_models optuna scikit-learn torch
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import roc_auc_score
from rtdl_revisiting_models import FTTransformer

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'; device

/home/flfp/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


'cpu'

## 1. Dados (sintéticos, binários)
Geramos 10 features numéricas e convertemos 3 delas em categóricas (por quantis) para testaros dois tipos de embedding.

In [2]:
X, y = make_classification(n_samples=6000, n_features=10, n_informative=6, random_state=0)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(10)])
cat_cols = ['f0', 'f1', 'f2']; num_cols = [c for c in df if c not in cat_cols]
for c in cat_cols: df[c] = pd.qcut(df[c], q=[5, 8, 4][cat_cols.index(c)], labels=False)

tr, te, ytr, yte = train_test_split(df, y, test_size=0.2, random_state=0, stratify=y)
tr, va, ytr, yva = train_test_split(tr, ytr, test_size=0.2, random_state=0, stratify=ytr)

qt = QuantileTransformer(output_distribution='normal').fit(tr[num_cols])
def prep(d, y):
    return (torch.tensor(qt.transform(d[num_cols]), dtype=torch.float32, device=device),
            torch.tensor(d[cat_cols].values, dtype=torch.long, device=device),
            torch.tensor(y, dtype=torch.float32, device=device))
Dtr, Dva, Dte = prep(tr, ytr), prep(va, yva), prep(te, yte)
cat_cardinalities = [int(df[c].nunique()) for c in cat_cols]
cat_cardinalities

[5, 8, 4]

## 2. Hiperparâmetros do modelo
`FTTransformer(n_cont_features, cat_cardinalities, d_out, **backbone)`. Principais:

| Hiperparâmetro | O que controla | Faixa típica (paper) |
|---|---|---|
| `n_blocks` | profundidade (nº de blocos Transformer) | 1–6 |
| `d_block` | dimensão dos embeddings/tokens (largura) | 64–512 (múltiplo de `attention_n_heads`) |
| `attention_n_heads` | nº de cabeças de atenção | 8 (padrão) |
| `attention_dropout` | dropout nos pesos de atenção | 0–0.5 |
| `ffn_d_hidden` / `ffn_d_hidden_multiplier` | tamanho da camada oculta da FFN (use um dos dois) | multiplier 2/3–8/3 (ReGLU) |
| `ffn_dropout` | dropout dentro da FFN | 0–0.5 |
| `residual_dropout` | dropout nas conexões residuais | 0–0.2 |
| `ffn_activation` | `'ReGLU'` (padrão), `'ReLU'`... | — |

Do **otimizador/treino**: `lr` (1e-5–1e-3), `weight_decay` (1e-6–1e-3), `batch_size`, nº de épocas + *early stopping*.
A lib já fornece `FTTransformer.get_default_kwargs(n_blocks)` e `model.make_default_optimizer()` (AdamW com weight decay desligado em LayerNorm, bias e embeddings).

In [3]:
def make_model(**kw):
    return FTTransformer(n_cont_features=len(num_cols), cat_cardinalities=cat_cardinalities, d_out=1, **kw).to(device)

FTTransformer.get_default_kwargs(n_blocks=3)

{'n_blocks': 3,
 'd_block': 192,
 'attention_n_heads': 8,
 'attention_dropout': 0.2,
 'ffn_d_hidden': None,
 'ffn_d_hidden_multiplier': 1.3333333333333333,
 'ffn_dropout': 0.1,
 'residual_dropout': 0.0,
 '_is_default': True}

## 3. Loop de treino com early stopping
Retorna o melhor AUC de validação (usado como objetivo na busca).

In [4]:
@torch.no_grad()
def predict(model, D):
    model.eval(); xn, xc, _ = D
    return torch.cat([model(xn[i:i+1024], xc[i:i+1024]).squeeze(-1) for i in range(0, len(xn), 1024)])

def fit(model, opt, batch_size=128, epochs=50, patience=8):
    xn, xc, y = Dtr; best, best_state, bad = 0, None, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(xn), device=device)
        for i in range(0, len(xn), batch_size):
            idx = perm[i:i+batch_size]
            loss = F.binary_cross_entropy_with_logits(model(xn[idx], xc[idx]).squeeze(-1), y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        auc = roc_auc_score(Dva[2].cpu(), predict(model, Dva).cpu())
        if auc > best: best, bad = auc, 0; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); return best

## 4. Baseline: configuração padrão do paper

In [5]:
model = make_model(**FTTransformer.get_default_kwargs(n_blocks=3))
print('params:', sum(p.numel() for p in model.parameters()))
print('val AUC:', fit(model, model.make_default_optimizer()))
print('test AUC:', roc_auc_score(yte, predict(model, Dte).cpu()))

params: 898369


val AUC: 0.9887282496885837
test AUC: 0.9830527307020296


## 5. Efeito de um hiperparâmetro isolado
Varremos `n_blocks` (os demais seguem o padrão correspondente).

In [6]:
rows = []
for nb in [1, 2, 3, 4]:
    torch.manual_seed(0)
    m = make_model(**FTTransformer.get_default_kwargs(nb))
    rows.append((nb, sum(p.numel() for p in m.parameters()), fit(m, m.make_default_optimizer())))
pd.DataFrame(rows, columns=['n_blocks', 'params', 'val_auc'])

,n_blocks,params,val_auc
0,1,78305,0.990304
1,2,269225,0.990929
2,3,898369,0.988728
3,4,2117289,0.992661


## 6. Busca de hiperparâmetros com Optuna
Espaço de busca inspirado no do paper (Tabela 14). `d_block` é escolhido entre múltiplos de `n_heads`.

In [7]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    torch.manual_seed(0)
    heads = 8
    kw = dict(
        n_blocks=trial.suggest_int('n_blocks', 1, 4),
        d_block=heads * trial.suggest_int('d_block_mult', 8, 48, step=8),   # 64..384
        attention_n_heads=heads,
        attention_dropout=trial.suggest_float('attention_dropout', 0.0, 0.5),
        ffn_d_hidden=None,
        ffn_d_hidden_multiplier=trial.suggest_float('ffn_mult', 2/3, 8/3),
        ffn_dropout=trial.suggest_float('ffn_dropout', 0.0, 0.5),
        residual_dropout=trial.suggest_float('residual_dropout', 0.0, 0.2),
    )
    model = make_model(**kw)
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    opt = torch.optim.AdamW(model.make_parameter_groups(), lr=lr, weight_decay=wd)
    return fit(model, opt, batch_size=trial.suggest_categorical('batch_size', [64, 128, 256]), epochs=30, patience=5)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=15)   # aumente para uma busca real
print(study.best_value); study.best_params

0.991770797616309


{'n_blocks': 3,
 'd_block_mult': 40,
 'attention_dropout': 0.10519128053692045,
 'ffn_mult': 0.9245192619763732,
 'ffn_dropout': 0.15771417546209193,
 'residual_dropout': 0.07274215418852452,
 'lr': 0.00013816356822574785,
 'weight_decay': 2.0692098656538357e-05,
 'batch_size': 64}

## 7. Retreinar com o melhor conjunto e avaliar no teste

In [8]:
p = study.best_params; torch.manual_seed(0)
model = make_model(n_blocks=p['n_blocks'], d_block=8 * p['d_block_mult'], attention_n_heads=8,
                   attention_dropout=p['attention_dropout'], ffn_d_hidden=None,
                   ffn_d_hidden_multiplier=p['ffn_mult'], ffn_dropout=p['ffn_dropout'],
                   residual_dropout=p['residual_dropout'])
opt = torch.optim.AdamW(model.make_parameter_groups(), lr=p['lr'], weight_decay=p['weight_decay'])
print('val AUC :', fit(model, opt, batch_size=p['batch_size'], epochs=50, patience=8))
print('test AUC:', roc_auc_score(yte, predict(model, Dte).cpu()))

val AUC : 0.991770797616309
test AUC: 0.9847666243517342


## Dicas :)
- Troque a seção 1: numéricas → `QuantileTransformer`/padronização; categóricas → inteiros (`0..cardinalidade-1`) e passe as cardinalidades em `cat_cardinalities`.
- Churn costuma ser desbalanceado: use `pos_weight` em `binary_cross_entropy_with_logits` e otimize AUC ou F1 em vez de acurácia.
- Compare com outros modelos baseline (XGBoost, MLP, etc) usando a mesma configuração.

Se precisar de ajuda, pode nos contatar:
- [Fábio Papais](https://github.com/fabiopapais): 81 98139-1034
- [Welton Felix](https://github.com/weltonfelix): 81 98443-4580